# Chapter 12: Natural Language Processing
**Part IV — Specialist Domains**

*The Complete MSc Reference: Artificial Intelligence & Machine Learning*  
*Jijesh Puliyappottammal — Digichange Publication, 2026*

---

NLP pipeline, tokenisation, transformers for text classification, RAG pipelines, and agentic LLMs.

## Learning Objectives
See Chapter 12 in the main textbook for full learning objectives, theory, and exam guidance.

## How to Use These Notebooks
Run cells from top to bottom. All cells are self-contained. Install any missing packages with `pip install <package>` in a new cell.


In [ ]:
# Standard imports used throughout the book
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

# Verify Python and key package versions
import sys
print("Python:", sys.version.split()[0])
try:
    import numpy, sklearn, torch
    print("NumPy:", numpy.__version__)
    print("Scikit-learn:", sklearn.__version__)
    print("PyTorch:", torch.__version__)
except ImportError as e:
    print(f"Missing: {e.name} — run: pip install {e.name}")


## Minimal RAG pipeline with sentence-transformers and FAISS


In [ ]:
# Minimal RAG pipeline with sentence-transformers and FAISS
from sentence_transformers import SentenceTransformer
import faiss, numpy as np

encoder = SentenceTransformer("all-MiniLM-L6-v2")

# Index documents
documents = ["AI is transforming healthcare.",
             "Machine learning requires data.",
             "LLMs power modern chatbots."]
doc_embeddings = encoder.encode(documents, normalize_embeddings=True)
index = faiss.IndexFlatIP(doc_embeddings.shape[1])
index.add(doc_embeddings.astype(np.float32))

# Retrieve top-3 for a query
query = "How is AI used in medicine?"
q_emb = encoder.encode([query], normalize_embeddings=True)
scores, ids = index.search(q_emb.astype(np.float32), k=3)
for i, s in zip(ids[0], scores[0]):
    print(f"[{s:.3f}] {documents[i]}")

## Load pre-trained ResNet-50


In [ ]:
from torchvision import models, transforms
from PIL import Image
import torch

# Load pre-trained ResNet-50
model = models.resnet50(weights="IMAGENET1K_V2")
model.eval()

# Preprocessing pipeline
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Predict
img = Image.open("cat.jpg")
tensor = preprocess(img).unsqueeze(0)
with torch.no_grad():
    logits = model(tensor)
probs = logits.softmax(1)
print("Top prediction:", probs.argmax().item())

## Byte Pair Encoding (BPE) — simple demo implementation


In [ ]:
# Byte Pair Encoding (BPE) — simple demo implementation
from collections import defaultdict, Counter

def get_vocab(corpus):
    """Convert corpus to character vocabulary with end-of-word marker."""
    vocab = defaultdict(int)
    for word in corpus.lower().split():
        vocab[' '.join(list(word)) + ' </w>'] += 1
    return vocab

def get_pairs(vocab):
    """Get all adjacent pair frequencies."""
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i+1])] += freq
    return pairs

def merge_vocab(pair, vocab):
    """Merge most frequent pair."""
    new_vocab = {}
    bigram = ' '.join(pair)
    replacement = ''.join(pair)
    for word, freq in vocab.items():
        new_vocab[word.replace(bigram, replacement)] = freq
    return new_vocab

corpus = "low lower lowest newer newest"
vocab = get_vocab(corpus)
print("Initial vocabulary:")
for k, v in sorted(vocab.items()):
    print(f"  {k!r}: {v}")

print("\nBPE merges:")
for step in range(8):
    pairs = get_pairs(vocab)
    if not pairs: break
    best = max(pairs, key=pairs.get)
    vocab = merge_vocab(best, vocab)
    print(f"  Step {step+1}: merge {best} → {''.join(best)} (freq={pairs[best]})")

print("\nFinal vocabulary:")
for k, v in sorted(vocab.items()):
    print(f"  {k!r}: {v}")

---

## 📚 Review Questions

See Chapter 12 of the textbook for:
- 5 review questions
- Common exam question with model answer (Appendix C)
- Flashcard summary (Appendix A)
- Formula sheet (Appendix B)

## 📖 Further Reading

See the Further Reading section at the end of Chapter 12 in the textbook.

---
*© 2026 Jijesh Puliyappottammal — Digichange Publication. Code examples are released under the MIT Licence for educational use.*